# S&P 500 10-Q Sentiment Analysis

### Exploring what large U.S. corporations are actually worried about

---

**Author:** Francesco Giordano  
**Date:** 2026  
**Dataset:** 50 random S&P 500 companies, most recent 10-Q filings  
**Source:** SEC EDGAR API

---

### The question

Every quarter, S&P 500 companies file a 10-Q with the SEC describing their financial performance and the risks they see ahead. Reading all 500 would take weeks. Instead, this notebook takes a random sample of 50 companies, downloads their latest 10-Q, and scores the narrative text across eight themes.

The goal is to answer: **what keeps corporate America up at night this quarter?**

### Approach

Rather than simple keyword counting, each keyword is assigned a directional weight (+2 strong concern, +1 mild concern, -1 mild positive, -2 strong positive), and a negation detection mechanism inverts the sign when a keyword is preceded by phrases like "no signs of" or "do not expect".

### Disclaimer

This notebook is a personal educational project. It is not financial, investment, or legal advice. See the [main repository](../README.md) for the full disclaimer.

---

## 1. Setup

Import the analysis module and configure the environment.

In [ ]:
import sys
import os

# Add the src/ directory to the Python path so we can import analyzer.py
sys.path.insert(0, os.path.abspath("../src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from analyzer import (
    load_sp500,
    load_cik_map,
    run_analysis,
    plot_results,
    plot_dashboard,
    TOPIC_KEYWORDS,
    AI_KEYWORDS,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Modules loaded successfully.")
print(f"Themes analyzed: {list(TOPIC_KEYWORDS.keys())}")
print(f"AI keywords tracked: {len(AI_KEYWORDS)}")

---

## 2. Data collection

### 2.1 Load the S&P 500 constituent list

The list is pulled from Wikipedia and then randomly sampled down to 50 companies for manageability.

In [ ]:
sp500 = load_sp500(sample_size=50, random_state=42)
print(f"Sampled {len(sp500)} companies:")
sp500.head(10)

### 2.2 Load the ticker-to-CIK mapping

The SEC identifies every company by a 10-digit CIK (Central Index Key), not by ticker. We need to translate.

In [ ]:
cik_map = load_cik_map()
print(f"Loaded {len(cik_map):,} ticker/CIK pairs from SEC.")
print(f"Example — AAPL: {cik_map.get('AAPL')}")

### 2.3 Run the full analysis

For each company, this step:
1. Finds the latest 10-Q
2. Downloads the main filing document
3. Cleans the HTML (stripping XBRL wrappers, tables, scripts)
4. Scores the text across all eight themes
5. Computes the AI bubble score

Expected runtime: 3-5 minutes (the script respects the SEC rate limit of 10 requests/sec).

In [ ]:
df = run_analysis(sp500, cik_map)
print(f"\nAnalysis complete. {len(df)} companies successfully processed.")

### 2.4 Preview the data

Each row is a company, each column is a theme score. Positive values = concern, negative = denial or positive framing.

In [ ]:
topic_cols = list(TOPIC_KEYWORDS.keys())
df[topic_cols + ['company', 'filingDate']].head(10)

---

## 3. First insights

### 3.1 Which themes dominate corporate discourse?

Let us aggregate the scores across all 50 companies to see which themes are mentioned most.

In [ ]:
totals = df[topic_cols].sum().sort_values(ascending=False)
totals.to_frame("total_score")

**Your observation:**

> Look at the ranking above. Which theme dominates? Is it what you expected? Does supply chain beat recession? Does inflation still rank higher than interest rates? Write a sentence or two here describing what surprises you in the results.

_For example: "Supply chain concerns dominate the sample, appearing more than 1.4x as often as recession. This suggests management is still focused on operational resilience after the post-COVID disruptions, even as macro narratives shift toward cyclical concerns."_

---

## 4. The big picture — composite dashboard

A single view combining:
- **Top:** per-company heatmap across all themes
- **Bottom-left:** aggregate theme ranking
- **Bottom-right:** top 20 companies by AI Bubble Score

In [ ]:
plot_dashboard(df)

---

## 5. The recession paradox

One of the most interesting features of the weighted + negation scoring is that **companies can score negative** on a theme — meaning they are actively **denying** the concern, or framing it positively. Let us isolate this.

In [ ]:
recession_deniers = df[df['recession'] < 0][['company', 'recession', 'filingDate']]
recession_deniers = recession_deniers.sort_values('recession')
print(f"Companies that score negative on recession: {len(recession_deniers)}")
recession_deniers

**Your observation:**

> A negative recession score is rare and meaningful. It indicates the company is going out of its way to describe a benign economic environment. Is this a self-fulfilling prophecy or a forward-looking signal? Note any pattern you see (a specific sector? specific CEOs?).

_For example: "The three companies with the most negative recession scores are all in consumer discretionary, which is counterintuitive — one might expect them to be the most exposed. This could indicate either strong internal data on consumer resilience, or simply defensive messaging to investors."_

---

## 6. Who talks about AI, and how?

The AI bubble score splits AI language into three buckets:
- **Bubble signals** — hype words, unquantified capex, AI race rhetoric
- **Solidity signals** — measurable ROI, governance, risk acknowledgment
- **Generic mentions** — neutral AI references

A high positive score = lots of hype, few concrete details.  
A negative score = the company discusses AI in terms of measurable outcomes and controls.

In [ ]:
ai_cols = ['ai_bubble_score', 'ai_bubble_hits', 'ai_solid_hits', 'ai_generic_hits']
ai_leaders = df.nlargest(15, 'ai_bubble_score')[ai_cols + ['company']]
ai_leaders

**Your observation:**

> Which companies top the AI hype ranking? Are they the ones you would expect (hyperscalers, chip makers)? Or are there surprises (e.g. a non-tech company with a high score)?

_For example: "The top 5 AI-bubble companies are dominated by chip makers and hyperscalers, which is expected. However, the presence of [company X] in the top 10 — a company in [sector Y] — is worth noting. Its score is driven by repeated mentions of 'AI transformation' without corresponding ROI language, a pattern typical of early-stage adoption."_

In [ ]:
# Ratio analysis: for every bubble signal, how many solidity signals?
df_ai = df[df['ai_bubble_hits'] > 0].copy()
df_ai['bubble_to_solid_ratio'] = df_ai['ai_bubble_hits'] / (df_ai['ai_solid_hits'] + 1)
df_ai = df_ai.sort_values('bubble_to_solid_ratio', ascending=False)

print("Companies with the highest hype-to-substance ratio:")
df_ai[['company', 'ai_bubble_hits', 'ai_solid_hits', 'bubble_to_solid_ratio']].head(10)

**Your observation:**

> A high ratio means many hype words with few concrete substantiations. Look at the top of this list — are these the usual suspects, or surprises? Is there a pattern by industry or by company size?

---

## 7. Do themes cluster together?

Are companies worried about inflation also worried about interest rates? Does recession concern correlate with consumer concern? A correlation matrix reveals hidden structure.

In [ ]:
corr_matrix = df[topic_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True, fmt=".2f",
    cmap="RdBu_r", center=0,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Pearson correlation'},
    ax=ax
)
ax.set_title("Theme correlation matrix", fontsize=13, fontweight='bold', pad=14)
plt.tight_layout()
plt.show()

**Your observation:**

> Which theme pairs show the strongest positive correlation? Which show negative correlation? Do you see any intuitive clusters (e.g., inflation + interest rates + credit) or counterintuitive ones?

---

## 8. The most concerned companies

Summing all theme scores gives a crude but useful "total concern" indicator. Which companies have the highest cumulative concern across themes?

In [ ]:
df['total_concern'] = df[topic_cols].sum(axis=1)
top_concerned = df.nlargest(15, 'total_concern')[['company', 'total_concern'] + topic_cols]
top_concerned

**Your observation:**

> Which industries or business models dominate the top of the list? Is there a pattern (banks focused on credit + interest rates, retailers on consumer, industrials on supply chain)?

---

## 9. Limitations of this analysis

Every exploratory analysis comes with caveats, and this one has several important ones:

1. **Keyword approach cannot handle complex semantics.** A sentence like "we had previously worried about recession, but now see stabilization" may be scored inconsistently depending on which keywords fire and within what window.

2. **50 companies is a small sample.** The random seed fixes reproducibility, but the sample may over-represent some sectors purely by chance. Expanding to the full 500 would likely smooth out sector biases.

3. **Dictionary bias.** The keyword lists reflect the author's judgment about what constitutes a "concern signal". Another analyst might weight "rate hike" differently, or include terms this analysis misses.

4. **No sector normalization.** A bank will naturally mention "credit risk" and "interest rates" more than a consumer goods company, regardless of whether those are elevated concerns. Proper peer comparison would require normalizing by sector averages.

5. **English only.** All keywords are English. Multilingual filings (rare in 10-Q, but possible in 20-F for foreign issuers) are not supported.

### Possible extensions

- Replace the keyword engine with a fine-tuned transformer like **FinBERT** for semantic understanding
- Run the analysis across multiple quarters to track sentiment drift over time
- Aggregate by GICS sector for peer comparison
- Add **Item 1A (Risk Factors)** to the analyzed text, not only the MD&A narrative

---

## 10. Key takeaways

**Write your three main findings here.** The goal is to distill the entire analysis into three clear, defensible observations that a reader could quote.

1. _(Your first finding — e.g. "Supply chain concerns dominate the sample, outranking recession concerns by a factor of 1.4x.")_

2. _(Your second finding — e.g. "Of 50 companies, 7 actively deny recession risk, a counterintuitive signal of corporate confidence.")_

3. _(Your third finding — e.g. "AI language is dominated by hype terms rather than concrete ROI mentions in 80% of the sample.")_

---

### Reproducibility

All code and data are available at [github.com/YOUR_USERNAME/sp500-10q-sentiment-analyzer](https://github.com/YOUR_USERNAME/sp500-10q-sentiment-analyzer). The analysis uses a fixed random seed (`random_state=42`) to ensure the sample is deterministic, but results will vary over time as companies file new 10-Qs.

### Contact

Feedback, suggestions, and pull requests welcome.